# Stage 1: Multiple Encoders per Modality on mRNA Stability

This notebook extends the Stage 1 representation analysis from one frozen encoder per modality to several. It uses the same 1,000-row sample of `mRNA_Stability.csv`, the same sequence filters, and the same analyses as [Stage1_refactor.ipynb](Stage1_refactor.ipynb), so results for the three encoders that appear in both notebooks can be checked against each other.

As in that notebook, every model input derives from one coding sequence: DNA encoders read it in DNA letters, RNA encoders read it in RNA letters, and protein encoders read its translation. Differences between encoders therefore reflect pretraining corpora, tokenization, and model size rather than new biological information. [Stage1_GTEx.ipynb](Stage1_GTEx.ipynb) repeats the analyses on data in which the DNA input is genomic sequence distinct from the transcript and protein.

| Modality | Encoder | Checkpoint | Reason for inclusion |
|---|---|---|---|
| DNA | Nucleotide Transformer 500M human-ref | `InstaDeepAI/nucleotide-transformer-500m-human-ref` | DNA encoder in Stage1_refactor.ipynb |
| DNA | Nucleotide Transformer v2 100M multi-species | `InstaDeepAI/nucleotide-transformer-v2-100m-multi-species` | DNA encoder in [BioLangFusion](../Papers/BioLangFusion.pdf), Table 3 |
| DNA | DNABERT-2 | `zhihan1996/DNABERT-2-117M` | Multi-species DNA encoder with byte-pair-encoding tokens |
| RNA | RNA-FM | `multimolecule/rnafm` | RNA encoder in Stage1_refactor.ipynb and BioLangFusion |
| RNA | mRNA-FM | `multimolecule/mrnafm` | RNA-FM variant pretrained on messenger RNA, with one token per codon |
| Protein | ESM-2 35M | `facebook/esm2_t12_35M_UR50D` | Protein encoder in Stage1_refactor.ipynb |
| Protein | ESM-2 8M | `facebook/esm2_t6_8M_UR50D` | Protein encoder in BioLangFusion, Table 3 |
| Protein | ESM-2 150M | `facebook/esm2_t30_150M_UR50D` | Protein encoder in [IsoFormer](../Papers/Multi-Modal-Transfer-Learning.pdf) |

**Running the notebook.** Use `Code/` as the working directory, as for Stage1_refactor.ipynb. The first run downloads the checkpoints into the Hugging Face cache (about 5 GB in total, including the three encoders that Stage1_refactor.ipynb already uses) and saves each encoder's embeddings in `stage1_multimodel_embeddings/`. Later runs reuse a saved matrix when the sample, checkpoint, and token limit are unchanged. Nucleotide Transformer v2 and DNABERT-2 ship model code written for an earlier version of Transformers; `stage1_utils.py` applies the small compatibility changes they need and pins both to a fixed checkpoint revision.

## Research questions

- **Track A: Decodability.** For each encoder, how much of mRNA stability and of known sequence properties can a linear probe recover from its frozen embedding?
- **Track B: Cross-modal geometry.** For every pair of encoders, including two encoders of the same modality, how similarly do their embedding spaces organize the same sequences, before and after removing sequence composition?
- **Track C: Selected diagnostics.** For a few pairs chosen from the Track B results, do layer-wise similarity and attention follow the patterns found in Stage1_refactor.ipynb?

## Role in the project

Stage 1 characterizes the representations available from frozen encoders before alignment is defined or fusion methods are built. Two encoders of the same modality read the same input type, so their agreement is the reference against which agreement across modalities is read. Together with Stage1_GTEx.ipynb, these results test whether the cross-modal agreement found in Stage1_refactor.ipynb, most of which sequence composition reproduces, depends on the particular encoders chosen.

---

## Setup

### Imports

Shared Stage 1 functions come from `stage1_utils.py` in this directory, so every Stage 1 notebook computes each metric with the same implementation. Its analysis functions reproduce those defined in Stage1_refactor.ipynb.

In [ ]:
# Standard library
import itertools
import os

# Data handling and visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Shared Stage 1 functions
import stage1_utils as s1

### Analysis settings

The sample settings match Stage1_refactor.ipynb: 1,000 rows drawn with `SEED = 42` before filtering. `ENCODER_KEYS` selects encoders from the registry in `stage1_utils.py`, and its order (DNA, RNA, protein) sets the row and column order of every table. The Track C settings choose the encoder pairs for layer-wise CKA and the nucleotide encoders for the attention comparison. They are starting choices to revisit after the Track B results.

`DEVICE` selects a CUDA GPU when available, then an Apple Silicon GPU through PyTorch's MPS backend, and otherwise the CPU.

In [ ]:
SEED = 42
torch.manual_seed(SEED)

# Input data, relative to the Code directory
MAIN_CSV_PATH = "mRNA_Stability.csv"
SEQ_COLUMN = "Sequence"
LABEL_COLUMN = "Value"

# Pilot sample size before sequence filtering, as in Stage1_refactor.ipynb
N_ROWS = 1000

# Encoders from the stage1_utils registry; this order sets the row and column order of every table
ENCODER_KEYS = [
    "nt500m", "ntv2_100m", "dnabert2",   # DNA
    "rnafm", "mrnafm",                   # RNA
    "esm2_8m", "esm2_35m", "esm2_150m",  # protein
]
ENCODERS = [s1.ENCODERS[key] for key in ENCODER_KEYS]
LABELS = {e.key: e.label for e in ENCODERS}
MODALITY = {e.key: e.modality for e in ENCODERS}

# Saved embeddings for this notebook, and the snapshot saved by Stage1_refactor.ipynb
EMBEDDING_DIR = "stage1_multimodel_embeddings"
REFACTOR_SNAPSHOT = "stage1_main_embeddings.npz"

# Track C selections
LAYERWISE_PAIRS = [("ntv2_100m", "dnabert2"), ("rnafm", "mrnafm"), ("mrnafm", "esm2_150m")]
LAYERWISE_N_ROWS = 200
ATTENTION_ENCODERS = ["ntv2_100m", "mrnafm"]
ATTENTION_N_ROWS = 100

DEVICE = s1.select_device()
print("Using device:", DEVICE)

pd.DataFrame(
    [
        {"encoder": e.label, "modality": e.modality, "checkpoint": e.checkpoint,
         "input": e.alphabet, "token limit": e.max_len}
        for e in ENCODERS
    ]
).set_index("encoder")

### Sample

The main dataset is sampled and filtered exactly as in Stage1_refactor.ipynb: a row is retained when its length is a multiple of three, it starts with ATG or AUG, and it translates to at least five amino acids. `kept_seqs` holds the retained sequences in sample order. It is the input to every encoder and the grouping key for every probe, so identical sequences never fall on both sides of a cross-validation split.

In [ ]:
main_df = (
    pd.read_csv(MAIN_CSV_PATH)
    .dropna(subset=[SEQ_COLUMN])
    .sample(n=N_ROWS, random_state=SEED)
)
kept_seqs, labels, seq_lengths = s1.retained_rows(main_df, SEQ_COLUMN, LABEL_COLUMN)
print(
    f"Retained {len(kept_seqs)} of {len(main_df)} sampled rows "
    f"({len(set(kept_seqs))} distinct sequences)."
)

### Encoder loading check

Each encoder is loaded once and applied to the first retained sequence. The table records the embedding width, the first tokens of that sequence, the longest tokenized input in the sample, and how many retained sequences exceed the encoder's token limit. Truncation keeps the start of a sequence; the encoder never sees the remainder.

In [ ]:
check_rows = []
for encoder in ENCODERS:
    tokenizer, model = s1.load_encoder(encoder, DEVICE)
    example = s1.encoder_input(encoder, kept_seqs[0])
    vector = s1.embed(example, tokenizer, model, encoder.max_len, DEVICE)
    example_ids = tokenizer(example, truncation=True, max_length=encoder.max_len)["input_ids"]
    full_lengths = [len(tokenizer(s1.encoder_input(encoder, s))["input_ids"]) for s in kept_seqs]
    check_rows.append({
        "encoder": encoder.label,
        "embedding width": vector.shape[0],
        "first tokens": " ".join(tokenizer.convert_ids_to_tokens(example_ids[:5])),
        "longest input (tokens)": max(full_lengths),
        "sequences truncated": sum(n > encoder.max_len for n in full_lengths),
    })
    del model, tokenizer
    s1.free_memory()

pd.DataFrame(check_rows).set_index("encoder")

---

## Embeddings

### Embedding the sample with every encoder

Each retained sequence is converted to the encoder's input (DNA letters, RNA letters, or the translated protein), tokenized with truncation at the encoder's limit, and passed through the frozen encoder with gradients disabled. The final hidden states are averaged over all tokens, including special tokens, giving one vector per sequence, as in Stage1_refactor.ipynb.

Each encoder's matrix is saved in `EMBEDDING_DIR` with its checkpoint, revision, token limit, and a fingerprint of the ordered input sequences. A saved matrix is reused only when all of these match, so changing the sample or an encoder setting recomputes only the affected encoders.

In [ ]:
EMBEDDINGS = {}
for encoder in ENCODERS:
    X, reused = s1.embed_with_cache(encoder, kept_seqs, EMBEDDING_DIR, DEVICE)
    EMBEDDINGS[encoder.key] = X
    status = "loaded from saved file" if reused else "computed and saved"
    print(f"{encoder.label}: {X.shape[0]} x {X.shape[1]} ({status})")

### Consistency with Stage1_refactor.ipynb

Nucleotide Transformer 500M, RNA-FM, and ESM-2 35M appear in both notebooks and read the same 981 sequences. Their embeddings here are compared with the snapshot that Stage1_refactor.ipynb saves in `stage1_main_embeddings.npz`. Differences near floating-point precision are expected when the two runs use different devices. Larger differences indicate a change in the sample, the code, or the software environment, which must be resolved before the new results are interpreted. Their stability probe results in Track A should also match those reported in Stage1_refactor.ipynb.

In [ ]:
SHARED_WITH_REFACTOR = {"nt500m": "dna", "rnafm": "rna", "esm2_35m": "protein"}

if os.path.exists(REFACTOR_SNAPSHOT):
    snapshot = np.load(REFACTOR_SNAPSHOT)
    print("Labels match:", np.allclose(snapshot["labels"], labels, rtol=0, atol=1e-9))
    for key, snapshot_key in SHARED_WITH_REFACTOR.items():
        ours, theirs = EMBEDDINGS[key], snapshot[snapshot_key]
        if ours.shape != theirs.shape:
            print(f"{LABELS[key]}: shapes differ, {ours.shape} versus {theirs.shape}")
            continue
        print(
            f"{LABELS[key]}: largest absolute difference {np.abs(ours - theirs).max():.2e} "
            f"(largest absolute value {np.abs(theirs).max():.2f})"
        )
else:
    print(f"{REFACTOR_SNAPSHOT} not found; run Stage1_refactor.ipynb to create it.")

---

## Representation analysis

### Track A: Decodability

#### Stability probes

A linear probe is a standardized ridge regression fit on one frozen embedding and evaluated with five-fold `GroupKFold`, grouping identical sequences. It measures how much of a target is linearly accessible without changing the encoder; the tables report the mean and standard deviation of the five held-out $R^2$ values (see *Linear probing* in Stage1_refactor.ipynb). The length-only baseline predicts stability from sequence length alone.

In [ ]:
NAMED_EMBEDDINGS = {LABELS[k]: EMBEDDINGS[k] for k in ENCODER_KEYS}

length_scores = s1.probe_scores(seq_lengths, labels, groups=kept_seqs)
print(f"Length-only baseline: R^2 {length_scores.mean():.3f} +/- {length_scores.std():.3f}")

stability_probes = s1.probe_table(NAMED_EMBEDDINGS, {"Stability": labels}, groups=kept_seqs)
stability_probes.insert(0, "modality", [MODALITY[k] for k in ENCODER_KEYS])
stability_probes.round(3)

#### Sequence-property probes

Log length, GC content, and GC3 are computable from the coding sequence, so these probes test whether each embedding retains properties of its input. GC3, the GC fraction at the third codon position, reflects synonymous codon choice, which translation removes. Protein encoders are therefore expected to recover GC3 less well than nucleotide encoders.

In [ ]:
seq_gc = np.array([s1.gc_content(s) for s in kept_seqs])
seq_gc3 = np.array([s1.gc3_content(s) for s in kept_seqs])
seq_log_length = np.log(seq_lengths.ravel())

property_probes = s1.probe_table(
    NAMED_EMBEDDINGS,
    {"Log length": seq_log_length, "GC": seq_gc, "GC3": seq_gc3},
    groups=kept_seqs,
)
property_probes.round(3)

#### Concatenated embeddings

For every pair of encoders, the probe is refit on the two embeddings placed side by side. The first matrix gives each concatenation's held-out $R^2$, and the second its difference from the better of the two encoders alone. A positive difference indicates linearly accessible stability information that neither encoder provides by itself. The table then concatenates all encoders of each modality, and all eight together.

In [ ]:
single_r2 = {k: stability_probes.loc[LABELS[k], "Stability R2"] for k in ENCODER_KEYS}


def concatenated_scores(keys):
    # Held-out probe scores for the listed encoders' embeddings placed side by side.
    return s1.probe_scores(np.hstack([EMBEDDINGS[k] for k in keys]), labels, groups=kept_seqs)


pair_concat = s1.pairwise_matrix(ENCODER_KEYS, lambda a, b: concatenated_scores([a, b]).mean())
pair_gain = s1.pairwise_matrix(
    ENCODER_KEYS, lambda a, b: pair_concat.loc[a, b] - max(single_r2[a], single_r2[b])
)
s1.plot_matrices(
    {"Concatenated pair, R^2": pair_concat, "Gain over the better encoder": pair_gain},
    LABELS,
    diverging={"Gain over the better encoder"},
    fmt="{:.3f}",
)

group_rows = []
for name, keys in [
    ("All DNA encoders", [k for k in ENCODER_KEYS if MODALITY[k] == "DNA"]),
    ("All RNA encoders", [k for k in ENCODER_KEYS if MODALITY[k] == "RNA"]),
    ("All protein encoders", [k for k in ENCODER_KEYS if MODALITY[k] == "protein"]),
    ("All encoders", ENCODER_KEYS),
]:
    scores = concatenated_scores(keys)
    group_rows.append({
        "combination": name,
        "R2": scores.mean(),
        "SD": scores.std(),
        "best member alone": max(single_r2[k] for k in keys),
    })
pd.DataFrame(group_rows).set_index("combination").round(3)

---

### Track B: Cross-modal geometry

#### Pairwise similarity

Every metric is computed for all 28 pairs of encoders, 21 across modalities and 7 within one, using the definitions in Stage1_refactor.ipynb:

- **Linear CKA:** agreement of the two spaces' similarity structures over all sequences.
- **Mutual $k$-NN:** average overlap of each sequence's 10 nearest neighbors in the two spaces. Chance is about $10/(n-1)$.
- **CCA and retrieval:** PCA (50 components) and CCA (10 components) are fitted on a 70% training split and evaluated on the held-out 30%. The tables report the first canonical correlation, the mean of all ten, and Recall@1 for finding each held-out sequence's partner, querying with the encoder listed first in `ENCODER_KEYS`. Chance is $1/n_{\text{test}}$.
- **RSA:** Spearman correlation between the two spaces' pairwise correlation distances, with a Mantel permutation test (499 permutations, so the smallest attainable $p$ is 0.002).

Within-modality pairs show how much two encoders of the same input type agree. They are the reference for reading agreement across modalities.

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(kept_seqs)), test_size=0.3, random_state=SEED
)
PAIRS = list(itertools.combinations(ENCODER_KEYS, 2))

cca_results = {
    (a, b): s1.cca_retrieval(EMBEDDINGS[a], EMBEDDINGS[b], train_idx, test_idx, SEED)
    for a, b in PAIRS
}


def from_cca(extract):
    # Turn a summary of one pair's CCA result into a function of the two encoder keys.
    return lambda a, b: extract(cca_results[(a, b)])


similarity = {
    "Linear CKA": s1.pairwise_matrix(
        ENCODER_KEYS, lambda a, b: s1.linear_cka(EMBEDDINGS[a], EMBEDDINGS[b])
    ),
    "Mutual k-NN": s1.pairwise_matrix(
        ENCODER_KEYS, lambda a, b: s1.mutual_knn_alignment(EMBEDDINGS[a], EMBEDDINGS[b], k=10)
    ),
    "First canonical correlation": s1.pairwise_matrix(ENCODER_KEYS, from_cca(lambda r: r["correlations"][0])),
    "Mean canonical correlation": s1.pairwise_matrix(ENCODER_KEYS, from_cca(lambda r: r["correlations"].mean())),
    "Recall@1": s1.pairwise_matrix(ENCODER_KEYS, from_cca(lambda r: r["recall"][1])),
}
print(
    f"Chance levels: mutual k-NN about {10 / (len(kept_seqs) - 1):.4f}; "
    f"Recall@1 {1 / len(test_idx):.4f} ({len(test_idx)} held-out rows)."
)
s1.plot_matrices({name: similarity[name] for name in ["Linear CKA", "Mutual k-NN", "Recall@1"]}, LABELS)

In [ ]:
similarity["RSA"] = s1.pairwise_matrix(
    ENCODER_KEYS, lambda a, b: s1.compute_rsa(EMBEDDINGS[a], EMBEDDINGS[b])[0]
)
similarity["RSA Mantel p"] = s1.pairwise_matrix(
    ENCODER_KEYS, lambda a, b: s1.mantel_p_value(EMBEDDINGS[a], EMBEDDINGS[b], seed=SEED)
)

pair_similarity = s1.pair_table(similarity, PAIRS, LABELS, MODALITY)
pair_similarity.round(3)

#### Composition control

Each embedding is replaced by its residual from an ordinary least-squares fit on 87 standardized composition features: GC content, GC3, log length, 64 codon frequencies, and 20 amino-acid frequencies, as in Stage1_refactor.ipynb. The fit uses all retained rows, so 88 unrelated predictors, counting the intercept, would explain about $88/n$ of the variance by chance. CKA and Recall@1 are then recomputed on the residuals with the same PCA-CCA split. Agreement that survives the control is the part that letter, codon, and amino-acid counts do not reproduce linearly.

In [ ]:
composition = np.array([s1.composition_features(s) for s in kept_seqs])
RESIDUALS = {k: s1.residualize(X, composition) for k, X in EMBEDDINGS.items()}

n_predictors = composition.shape[1] + 1  # composition features plus the intercept
print(
    f"Chance level for {n_predictors} predictors on {len(kept_seqs)} rows: "
    f"about {n_predictors / len(kept_seqs):.3f}"
)
composition_shares = pd.Series(
    {LABELS[k]: s1.composition_share(EMBEDDINGS[k], RESIDUALS[k]) for k in ENCODER_KEYS},
    name="share of variance explained by composition",
)
display(composition_shares.round(3).to_frame())

after_control = {
    "Linear CKA after control": s1.pairwise_matrix(
        ENCODER_KEYS, lambda a, b: s1.linear_cka(RESIDUALS[a], RESIDUALS[b])
    ),
    "Recall@1 after control": s1.pairwise_matrix(
        ENCODER_KEYS,
        lambda a, b: s1.cca_retrieval(RESIDUALS[a], RESIDUALS[b], train_idx, test_idx, SEED)["recall"][1],
    ),
}
s1.plot_matrices(after_control, LABELS)

composition_control = s1.pair_table(
    {
        "CKA before": similarity["Linear CKA"],
        "CKA after": after_control["Linear CKA after control"],
        "Recall@1 before": similarity["Recall@1"],
        "Recall@1 after": after_control["Recall@1 after control"],
    },
    PAIRS, LABELS, MODALITY,
)
composition_control["CKA reduction"] = 1 - composition_control["CKA after"] / composition_control["CKA before"]
composition_control.round(3)

---

### Track C: Selected diagnostics

#### Layer-wise CKA for selected pairs

For each pair in `LAYERWISE_PAIRS`, every hidden state of both encoders is mean-pooled and CKA is computed for every combination of layers. Layer 0 is the embedding layer, before any attention. The `LAYERWISE_N_ROWS` sequences are drawn from the retained rows, so the subsample contains exactly that number. Linear CKA is inflated when the number of sequences is small relative to the embedding widths, so these values are compared within this analysis rather than with the full-sample Track B values.

In [ ]:
rng = np.random.default_rng(SEED)
layer_rows = np.sort(rng.choice(len(kept_seqs), size=min(LAYERWISE_N_ROWS, len(kept_seqs)), replace=False))

LAYER_MATRICES = {}
for key in sorted({k for pair in LAYERWISE_PAIRS for k in pair}, key=ENCODER_KEYS.index):
    encoder = s1.ENCODERS[key]
    tokenizer, model = s1.load_encoder(encoder, DEVICE)
    per_sequence = [s1.embed_all_layers(kept_seqs[i], encoder, tokenizer, model, DEVICE) for i in layer_rows]
    LAYER_MATRICES[key] = [
        np.vstack([states[layer] for states in per_sequence]) for layer in range(len(per_sequence[0]))
    ]
    del model, tokenizer
    s1.free_memory()
    print(f"{LABELS[key]}: {len(LAYER_MATRICES[key])} hidden states on {len(layer_rows)} sequences")

layer_cka = {}
for a, b in LAYERWISE_PAIRS:
    matrix = np.array([[s1.linear_cka(A, B) for B in LAYER_MATRICES[b]] for A in LAYER_MATRICES[a]])
    layer_cka[(a, b)] = matrix
    i, j = np.unravel_index(matrix.argmax(), matrix.shape)
    print(
        f"{LABELS[a]} vs {LABELS[b]}: maximum {matrix.max():.3f} at layers {i} and {j}; "
        f"final layers {matrix[-1, -1]:.3f}"
    )

vmax = max(matrix.max() for matrix in layer_cka.values())
fig, axes = plt.subplots(1, len(layer_cka), figsize=(6 * len(layer_cka), 5), squeeze=False)
for ax, ((a, b), matrix) in zip(axes[0], layer_cka.items()):
    im = ax.imshow(matrix, cmap="Greens", aspect="auto", origin="lower", vmin=0, vmax=vmax)
    ax.set_xlabel(f"{LABELS[b]} layer (0 = embedding layer)")
    ax.set_ylabel(f"{LABELS[a]} layer (0 = embedding layer)")
    ax.set_title(f"Layer-wise linear CKA, {LABELS[a]} vs {LABELS[b]}", fontsize=10)
fig.colorbar(im, ax=axes[0].tolist(), label="CKA")
plt.show()

#### Attention at candidate stability motifs

For each encoder in `ATTENTION_ENCODERS`, the last-layer attention each token receives is averaged over heads, min-max scaled, and repeated over the nucleotides the token spells: six for most Nucleotide Transformer tokens and three for mRNA-FM codons. A one-sided Mann-Whitney test compares scores at AU-rich-element and m$^6$A consensus matches with all other positions, separately for each of the first `ATTENTION_N_ROWS` retained sequences. These coding-only sequences contain few such elements, which concentrate in 3′ UTRs, so this comparison repeats the Stage1_refactor.ipynb analysis for the new nucleotide encoders rather than providing a strong test.

In [ ]:
attention_rows = []
for key in ATTENTION_ENCODERS:
    encoder = s1.ENCODERS[key]
    tokenizer, model = s1.load_encoder(encoder, DEVICE, eager_attention=True)
    results = []
    for seq in kept_seqs[:ATTENTION_N_ROWS]:
        attention = s1.nucleotide_attention(seq, encoder, tokenizer, model, DEVICE)
        result = s1.attention_motif_overlap(attention, s1.scan_sequence_for_motifs(seq))
        if result is not None:
            results.append(result)
    del model, tokenizer
    s1.free_memory()

    groups = {"All motifs": results}
    for motif in s1.STABILITY_MOTIFS:
        groups[motif] = [r["by_motif"][motif] for r in results if r["by_motif"].get(motif)]
    for motif, group in groups.items():
        row = {"encoder": LABELS[key], "motif": motif, "sequences with hits": len(group)}
        if group:
            row["fraction with p < 0.05"] = np.mean([g["p_value"] < 0.05 for g in group])
            row["mean gap"] = np.mean([g["mean_at_motifs"] - g["mean_elsewhere"] for g in group])
        attention_rows.append(row)

pd.DataFrame(attention_rows).set_index(["encoder", "motif"]).round(4)

---

## Summary

The summary records what the additional encoders change about the Stage 1 findings and what they imply for defining alignment. It is written after the analyses above have been run.